# 06. Economic Loss Analysis

This notebook computes **Energy Loss ($	ext{kWh}$)** and **Financial Loss ($	ext{₹}$)** for persistent solar PV anomalies using configurable tariff models.

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to sys.path
sys.path.append(str(Path("..").resolve()))

from src.config import CLEANED_DATA_PATH
from src.predict import predict_expected_power
from src.anomaly import detect_anomalies
from src.economics import calculate_economic_losses

sns.set_theme(style="whitegrid")

## 1. Run Economic Loss Pipeline

In [ ]:
df_clean = pd.read_csv(CLEANED_DATA_PATH)
df_pred = predict_expected_power(df_clean)
df_anom, thresh = detect_anomalies(df_pred)

# Compute losses with default tariff ₹8.0 / kWh
summary, df_econ = calculate_economic_losses(df_anom, tariff_inr_kwh=8.0)

print("--- ECONOMIC LOSS SUMMARY --- ")
for k, v in summary.items():
    print(f"{k}: {v}")

## 2. Tariff Sensitivity Analysis (Sensitivity from ₹4/kWh to ₹12/kWh)

In [ ]:
tariffs = [4.0, 6.0, 8.0, 10.0, 12.0]
results = []
for t in tariffs:
    s, _ = calculate_economic_losses(df_anom, tariff_inr_kwh=t)
    results.append({"Tariff (₹/kWh)": t, "Total Loss (₹)": s["Total Financial Loss (₹)"]})

df_sens = pd.DataFrame(results)
plt.figure(figsize=(8, 5))
plt.plot(df_sens["Tariff (₹/kWh)"], df_sens["Total Loss (₹)"], marker='o', color='#c0392b', linewidth=2)
plt.xlabel('Tariff Rate (₹/kWh)')
plt.ylabel('Total Financial Loss (₹)')
plt.title('Financial Loss Sensitivity Across Tariff Rates')
plt.tight_layout()
plt.show()